# Distributed Spam Detection

For presentational purposes I assume Ray is running on port 8080.

In [1]:
import ray
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
if ray.is_initialized:
    ray.shutdown()

ray.init()

2026-05-31 19:00:37,436	INFO worker.py:2012 -- Started a local Ray instance.
C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.12
Ray version:,2.55.1



## Data loading and preprocessing

In [3]:
import pandas as pd

local_path = r"data\enron_spam_data.csv"

pdf = pd.read_csv(
    local_path,
    on_bad_lines='skip',
    encoding='utf-8',
    low_memory=False
)

pdf = pdf.fillna({"Message": "", "Subject": ""})

df = ray.data.from_pandas(pdf)
df.show(5)

2026-05-31 19:00:51,591	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-05-31 19:00:51,605	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-05-31 19:00:51,624	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-05-31_19-00-33_565244_24632\logs\ray-data
2026-05-31 19:00:51,625	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> LimitOperator[limit=5]
2026-05-31 19:00:51,627	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (1.4GiB out of 3.3GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJE

{'Message ID': 0, 'Subject': 'christmas tree farm pictures', 'Message': '', 'Spam/Ham': 'ham', 'Date': '1999-12-10'}
{'Message ID': 1, 'Subject': 'vastar resources , inc .', 'Message': 'gary , production from the high island larger block a - 1 # 2 commenced on\nsaturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and\n10 , 000 gross for tomorrow . vastar owns 68 % of the gross production .\ngeorge x 3 - 6992\n- - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16\nam - - - - - - - - - - - - - - - - - - - - - - - - - - -\ndaren j farmer\n12 / 10 / 99 10 : 38 am\nto : carlos j rodriguez / hou / ect @ ect\ncc : george weissman / hou / ect @ ect , melissa graves / hou / ect @ ect\nsubject : vastar resources , inc .\ncarlos ,\nplease call linda and get everything set up .\ni \' m going to estimate 4 , 500 coming up tomorrow , with a 2 , 000 increase each\nfollowing day based on my conversations with bill fis

In [4]:
def preprocess_batch(batch):
    subject_clean = batch["Subject"].fillna("")
    message_clean = batch["Message"].fillna("")

    batch["Text"] = subject_clean + " " + message_clean

    batch["Spam/Ham"] = batch["Spam/Ham"].map({"ham": 0, "spam": 1})

    return batch

processed_df = df.map_batches(preprocess_batch, batch_format="pandas")

train, test = processed_df.train_test_split(test_size=0.2, shuffle=True)
print("Train set:")
train.show(1)
print("Test set:")
test.show(1)

2026-05-31 19:00:52,364	INFO logging.py:416 -- Registered dataset logger for dataset dataset_4_0
2026-05-31 19:00:52,366	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_4_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-05-31_19-00-33_565244_24632\logs\ray-data
2026-05-31 19:00:52,367	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_4_0: InputDataBuffer[Input] -> AllToAllOperator[MapBatches(preprocess_batch)->RandomShuffle]
2026-05-31 19:00:52,373	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_4_0 =======
2026-05-31 19:00:52,375	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-31 19:00:52,376	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/716.0MiB object store
2026-05-31 19:00:52,376	INFO logging_progress.py:181 -- 
2026-05-31 19:00:52,376	INFO logging_progress.py:231 -- MapBatches(preprocess_batch)->RandomShuffle: 0/1
2026-05-31 19:00:52,377	INFO logging_progress

Train set:
{'Message ID': 9537, 'Subject': 'your membership community charset = iso - 8859 - 1', 'Message': 'your membership community & commentary ( july 6 , 2001 )\nit \' s all about making money\ninformation to provide you with the absolute\nbest low and no cost ways of providing traffic\nto your site , helping you to capitalize on the power and potential the web brings to every net - preneur .\n- - - this issue contains sites who will trade links with you ! - - -\n- - - - - - - - - - - - -\nin this issue\n- - - - - - - - - - - - -\ninternet success through simplicity\nmember showcase\nwin a free ad in community & commentary\n| | | = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = > >\ntoday \' s special announcement :\n| | | = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = > >\nwe can help you become an internet service provider within 7 days or we will give you $ 100 . 00 ! !\nclick here\nwe have already signed 300 isps on a 4 year contract

## Training a model

In [5]:
import tensorflow as tf

def build_model() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(200,), dtype=tf.int32)
    x = tf.keras.layers.Embedding(input_dim=10000, output_dim=32)(inputs)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(100, activation="relu")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)

In [14]:
from ray.train.tensorflow.keras import ReportCheckpointCallback

def training_loop(config: dict):
    tf.keras.backend.clear_session()
    strategy = tf.distribute.MultiWorkerMirroredStrategy()

    batch_size = config.get("batch_size", 32)
    epochs = config.get("epochs", 4)
    X_global = config.get("X_global")
    y_global = config.get("y_global")

    with strategy.scope():
        model = build_model()
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    model.fit(X_global, y_global, batch_size=batch_size, epochs=epochs, callbacks=[ReportCheckpointCallback()])


In [10]:
global_vectorizer = tf.keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=200)
global_vectorizer.adapt(train.to_pandas()["Text"])

X_global = global_vectorizer(train.to_pandas()["Text"].astype(object)).numpy()
y_global = train.to_pandas()["Spam/Ham"].values

2026-05-31 19:04:26,076	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_10
2026-05-31 19:04:26,084	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_10 =======
2026-05-31 19:04:26,086	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-31 19:04:26,086	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/716.0MiB object store
2026-05-31 19:04:26,087	INFO logging_progress.py:192 -- =============================================
2026-05-31 19:04:26,097	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_6_10 execution finished in 0.03 seconds
2026-05-31 19:04:27,333	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_11
2026-05-31 19:04:27,341	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_11 =======
2026-05-31 19:04:27,343	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-31 19:04:27,343	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/716.

In [15]:
from ray.train.tensorflow import TensorflowTrainer

scaling_config = ray.train.ScalingConfig(num_workers=2, use_gpu=False)

trainer = TensorflowTrainer(
    train_loop_per_worker=training_loop,
    train_loop_config={"batch_size": 32, "epochs": 5, "X_global": X_global, "y_global": y_global},
    scaling_config=scaling_config
)
results = trainer.fit()

(TrainController pid=28388) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(TrainController pid=28388) I0000 00:00:1780247807.045877   25800 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(TrainController pid=28388) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(TrainController pid=28388) I0000 00:00:1780247808.486938   25800 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(TrainController pid=28388) WARNING:tensorflow:From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\l

(RayTrainWorker pid=31276) Epoch 1/5


(RayTrainWorker pid=31276) INFO:tensorflow:Collective all_reduce IndexedSlices: 1 all_reduces, num_devices =1, group_size = 2, implementation = CommunicationImplementation.AUTO
(RayTrainWorker pid=31276) Collective all_reduce IndexedSlices: 1 all_reduces, num_devices =1, group_size = 2, implementation = CommunicationImplementation.AUTO
(PlacementGroupCleaner pid=29852) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(RayTrainWorker pid=31276) WARNING:tensorflow:From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\engine\base_layer_utils.py:384: The name tf.executing_eagerly_outside_functions is deprecated. Please use tf.compat.v1.executing_eagerly_outside_functions instead.
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=31276) From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\engine\base_layer_utils.py:384: The name

(RayTrainWorker pid=31276) 
(RayTrainWorker pid=31276)   1/843 [..............................] - ETA: 10:44 - loss: 1.3857 - accuracy: 1.0625
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=31276)  29/843 [>.............................] - ETA: 6s - loss: 1.3762 - accuracy: 1.2306
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=31276)  57/843 [=>............................] - ETA: 6s - loss: 1.3512 - accuracy: 1.3213
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=29304) 


(PlacementGroupCleaner pid=29852) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(RayTrainWorker pid=29304) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(RayTrainWorker pid=29304) I0000 00:00:1780247828.101016   32144 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`. [repeated 3x across cluster]


...

(PlacementGroupCleaner pid=29852) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(RayTrainWorker pid=31276) 
(RayTrainWorker pid=31276) 510/843 [=================>............] - ETA: 2s - loss: 0.0177 - accuracy: 1.9946
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=31276) 538/843 [==================>...........] - ETA: 2s - loss: 0.0177 - accuracy: 1.9945
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=31276) 566/843 [===================>..........] - ETA: 2s - loss: 0.0177 - accuracy: 1.9946
(RayTrainWorker pid=31276) 573/843 [===================>..........] - ETA: 2s - loss: 0.0178 - accuracy: 1.9945
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=31276) 593/843 [====================>.........] - ETA: 2s - loss: 0.0181 - accuracy: 1.9943
(RayTrainWorker pid=

(PlacementGroupCleaner pid=29852) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(RayTrainWorker pid=29304) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/krzys/ray_results/ray_train_run-2026-05-31_19-16-39/checkpoint_2026-05-31_19-17-38.326890)
(RayTrainWorker pid=29304) Reporting training result 4: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=C:/Users/krzys/ray_results/ray_train_run-2026-05-31_19-16-39/checkpoint_2026-05-31_19-17-38.326890), metrics={'loss': 0.012586439028382301, 'accuracy': 0.9960328936576843}, validation=False)


(RayTrainWorker pid=31276) 
(RayTrainWorker pid=31276) 731/843 [=========================>....] - ETA: 0s - loss: 0.0186 - accuracy: 1.9938
(RayTrainWorker pid=31276) 737/843 [=========================>....] - ETA: 0s - loss: 0.0187 - accuracy: 1.9937
(RayTrainWorker pid=31276) 741/843 [=========================>....] - ETA: 0s - loss: 0.0189 - accuracy: 1.9937
(RayTrainWorker pid=29304) 165/843 [====>.........................] - ETA: 5s - loss: 0.0157 - accuracy: 1.9951 [repeated 9x across cluster]
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=31276) 761/843 [==========================>...] - ETA: 0s - loss: 0.0188 - accuracy: 1.9937
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pid=29304) 190/843 [=====>........................] - ETA: 5s - loss: 0.0167 - accuracy: 1.9944 [repeated 6x across cluster]
(RayTrainWorker pid=29304) 
(RayTrainWorker pid=31276) 
(RayTrainWorker pi

(RayTrainWorker pid=31276) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/krzys/ray_results/ray_train_run-2026-05-31_19-16-39/checkpoint_2026-05-31_19-17-45.604497)
(RayTrainWorker pid=31276) Reporting training result 5: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=C:/Users/krzys/ray_results/ray_train_run-2026-05-31_19-16-39/checkpoint_2026-05-31_19-17-45.604497), metrics={'loss': 0.009190445765852928, 'accuracy': 0.9968856573104858}, validation=False)
(PlacementGroupCleaner pid=29852) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


### Evaluation

In [17]:
print(f"Final results: {results.metrics}")

Final results: {'loss': 0.009190445765852928, 'accuracy': 0.9968856573104858}
